<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SETUP INIZIALE — Installa dipendenze e configura API
# ═══════════════════════════════════════════════════════════════════

# Installa le librerie necessarie in modalità silenziosa (-q = quiet)
# ├─ anthropic: SDK per Claude API
# ├─ chromadb: Database vettoriale per embedding semantici
# ├─ pypdf: Lettura di file PDF
# └─ sentence-transformers: Modelli per generare embedding da testo
!pip install anthropic chromadb pypdf sentence-transformers -q

# Importa il modulo Google Colab per recuperare le credenziali dall'environment
from google.colab import userdata

# Importa le librerie principali
import anthropic  # Client per API Claude
import os         # Per gestire variabili ambiente

# Recupera la chiave API da Google Colab secrets e la salva nell'environment
# Questo rende disponibile la chiave a tutti i client che la cercano
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Crea un client Anthropic con la chiave API configurata nell'environment
client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    """
    Funzione helper per fare domande a Claude in modo semplice.
    
    Parametri:
    - domanda: il testo della domanda dell'utente
    - system: (opzionale) istruzioni di sistema per il comportamento di Claude
    - max_tokens: numero massimo di token nella risposta (default=800)
    
    Restituisce: il testo della risposta di Claude
    """
    # Crea un dizionario con i parametri della richiesta
    params = {
        "model": "claude-haiku-4-5-20251001",  # Modello Claude da usare
        "max_tokens": max_tokens,               # Limita la lunghezza risposta
        "messages": [                           # Lista di messaggi della conversazione
            {"role": "user", "content": domanda}  # Il messaggio dell'utente
        ]
    }
    # Se è stato fornito un system prompt, aggiungilo ai parametri
    if system:
        params["system"] = system
    
    # Chiama l'API Claude e restituisce solo il testo della risposta
    return client.messages.create(**params).content[0].text

# Stampa un messaggio di conferma che il setup è completato
print("✅ Setup completato!")


✅ Setup completato!


---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CREA DOCUMENTO DI TEST — Un manuale WiData su sensori IoT
# ═══════════════════════════════════════════════════════════════════

# Definiamo un documento di testo con informazioni su WiData Srl
# (In un progetto reale useresti PDF veri caricati da file)
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva il documento in un file di testo per persisterenza
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    # "w" = modalità scrittura (crea file nuovo o sovrascrive se esiste)
    # encoding="utf-8" = salva con caratteri italiani (accenti, etc.)
    f.write(documento_widata)

# Stampa a video la lunghezza del documento
print(f"✅ Documento creato: {len(documento_widata)} caratteri")


✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CHUNKING — Dividi il documento in pezzi piccoli (chunks)
# ═══════════════════════════════════════════════════════════════════
# Perché? I modelli di embedding lavorano meglio con testi di lunghezza moderata
# Se il testo è troppo lungo, perde contesto e la similarità semantica è imprecisa

def chunka_testo(testo, chunk_size=400, overlap=50):
    """
    Divide un testo lungo in pezzi (chunk) più piccoli con sovrapposizione.
    
    Come funziona:
    - chunk_size: lunghezza di ogni pezzo (in caratteri)
    - overlap: quanti caratteri far sovrapporre tra un chunk e il successivo
    
    Esempio con overlap:
    "ABCDEFGHIJ" con chunk_size=4, overlap=2
    → Chunk 1: "ABCD"
    → Chunk 2: "CDEF"  (sovrappone "CD")
    → Chunk 3: "EFGH"  (sovrappone "EF")
    
    L'overlap serve a non perdere info ai confini tra chunk
    """
    chunks = []                              # Lista dove salviamo i chunk
    start = 0                                # Posizione iniziale nel testo
    
    # Continua finché non raggiungi la fine del testo
    while start < len(testo):
        # Calcola la posizione finale di questo chunk
        end = start + chunk_size
        
        # Estrai il pezzo di testo da start a end
        chunk = testo[start:end]
        
        # Se il chunk non è vuoto (cioè non è solo spazi), aggiungilo alla lista
        if chunk.strip():  # .strip() rimuove spazi e newline
            chunks.append(chunk)
        
        # Sposta start in avanti di (chunk_size - overlap)
        # Così il prossimo chunk avrà overlap caratteri in comune
        start += chunk_size - overlap
    
    return chunks


# Applica la funzione al documento WiData
chunks = chunka_testo(documento_widata)

# Stampa il numero totale di chunk creati
print(f"📊 Numero di chunk: {len(chunks)}")
print()

# Per ogni chunk, mostra una preview
for i, chunk in enumerate(chunks):
    # Numero del chunk + lunghezza in caratteri
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    
    # I primi 100 caratteri del chunk + "..." per indicare che continua
    print(chunk[:100]+"...")
    print()


📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G ...

--- Chunk 4 (400 char) ---
iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-tim...

--- Chunk 5 (400 char) ---
va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiest...

--- Chunk 6 (96 char) ---
: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INDICIZZAZIONE — Salva i chunk in ChromaDB con embedding vettoriali
# ═══════════════════════════════════════════════════════════════════
# ChromaDB crea automaticamente gli embedding usando sentence-transformers

import chromadb

# Crea un client ChromaDB che salva i dati in memoria (non su disco)
# (Per produzione, useresti:  chromadb.PersistentClient(path="./db"))
chroma_client = chromadb.Client()

# Crea una nuova "collection" (tabella/indice) per i documenti WiData
# Oppure la recupera se esiste già
collection = chroma_client.get_or_create_collection(
    name="widata_docs",  # Nome della collection
    metadata={
        "hnsw:space": "cosine"  # Usa la distanza coseno per similarità semantica
        # (alternativa: "l2" per distanza euclidea)
    }
)

# Aggiungi tutti i chunk alla collection
collection.add(
    documents=chunks,                                    # Lista dei chunk di testo
    ids=[f"chunk_{i}" for i in range(len(chunks))]      # ID unico per ogni chunk
    # ChromaDB calcolerà automaticamente gli embedding per ogni documento
)

# Stampa il numero di chunk indicizzati
print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")

# Spiegazione di cosa è successo dietro le quinte:
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")
# ChromaDB:
# 1. Ha preso ogni chunk di testo
# 2. Lo ha passato a sentence-transformers (modello di embedding)
# 3. Ha generato un vettore di 384 dimensioni per ogni chunk
# 4. Ha salvato il vettore in un indice HNSW (Hierarchical Navigable Small World)
# 5. Ora può trovare velocemente chunk simili usando la distanza coseno


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 98.2MiB/s]


✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RICERCA SEMANTICA — Trova i chunk più rilevanti per una domanda
# ═══════════════════════════════════════════════════════════════════

def cerca(domanda, n_risultati=3):
    """
    Cerca i chunk più simili a una domanda usando la similarità coseno.
    
    Come funziona:
    1. ChromaDB genera l'embedding della domanda (stesso modello dei chunk)
    2. Calcola la similarità coseno tra la domanda e ogni chunk
    3. Ritorna i top-n chunk più simili
    """
    # Chiama il metodo query() di ChromaDB
    risultati = collection.query(
        query_texts=[domanda],          # La domanda di ricerca
        n_results=n_risultati           # Numero di chunk da ritornare
    )
    
    # collection.query() ritorna un dizionario con struttura:
    # {"documents": [[chunk1, chunk2, ...]], "ids": [...], "distances": [...], ...}
    # Estraiamo solo i documenti (testi) del primo (e unico) risultato
    return risultati["documents"][0]


# Prepariamo alcune domande di test
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

# Per ogni domanda, mostra i chunk recuperati
for domanda in domande_test:
    print(f"\n❓ {domanda}")
    
    # Cerca i 2 chunk più rilevanti
    chunks_trovati = cerca(domanda, n_risultati=2)
    
    # Mostra ogni chunk trovato
    for i, chunk in enumerate(chunks_trovati):
        # Mostra il numero e i primi 120 caratteri
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")



❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RAG COMPLETO — Retrieval + Augmentation + Generation
# ═══════════════════════════════════════════════════════════════════
# Questo è il cuore della pipeline RAG:
# 1. Retrieval: recupera i chunk rilevanti dalla base di conoscenza
# 2. Augmentation: arricchisci il prompt con i chunk
# 3. Generation: chiedi a Claude di rispondere basandosi su quel contesto

# System prompt per Claude — spiega il suo ruolo e come comportarsi
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""
# Questa istruzione "anti-hallucination" previene che Claude inventi informazioni
# non presenti nei documenti

def chat_rag(domanda, n_chunks=3):
    """
    Implementa la pipeline RAG completa per una domanda.
    
    Flusso:
    1. Recupera i chunk rilevanti (RETRIEVAL)
    2. Costruisci un prompt aumentato con il contesto (AUGMENTATION)
    3. Chiedi a Claude di rispondere (GENERATION)
    """
    
    # STEP 1: RETRIEVAL
    # Cerca i chunk rilevanti per la domanda
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    
    # Unisci i chunk in un unico testo di contesto
    # Separa i chunk con "---" per chiarezza
    contesto = "\n\n---\n\n".join(chunks_rilevanti)
    
    # STEP 2: AUGMENTATION
    # Costruisci il prompt che manderai a Claude
    # Include il contesto recuperato + la domanda
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda dell'utente: {domanda}"""
    
    # STEP 3: GENERATION
    # Chiama Claude con il system prompt e il prompt aumentato
    # Claude avrà il contesto necessario per rispondere accuratamente
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    
    # Restituisci sia la risposta che i chunk usati (per trasparenza)
    return risposta, chunks_rilevanti


# Test della pipeline RAG
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"

# Chiama la funzione RAG
risposta, chunks = chat_rag(domanda)

# Stampa i risultati
print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")


❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 Sì, il sensore XS200 è in grado di funzionare in ambienti freddi. 

Secondo le specifiche tecniche, il sensore misura temperature da **-20°C a +60°C**, quindi può operare correttamente anche in condizioni di freddo intenso fino a -20°C.

Inoltre, grazie alla classificazione **IP67**, il sensore è impermeabile e resistente alla polvere, il che lo rende adatto anche ad ambienti difficili dal punto di vista climatico.

📄 Basato su 3 chunk


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TEST DI CONTROLLO — Domanda fuori dalla base di conoscenza
# ═══════════════════════════════════════════════════════════════════
# Verifichiamo che il sistema rispetti l'istruzione "anti-hallucination"
# e rifiuti di rispondere a domande non coperte dai documenti

# Fai una domanda che NON è nei documenti WiData
domanda_off = "Quali sono i migliori smartphone del 2025?"

# Chiama la funzione RAG (anche se la domanda è fuori contesto)
risposta_off, _ = chat_rag(domanda_off)

# Stampa la risposta
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")
# Se il system prompt funziona, Claude dirà "Non ho questa informazione nei miei documenti"
# Se dovesse inventare una risposta, il system prompt non è stato rispettato


❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano esclusivamente i servizi e prodotti di WiData Srl, come piani di abbonamento, sensori IoT, gateway e piattaforme di analisi dati. Non contengono informazioni su smartphone.

Posso aiutarti con domande relative ai nostri prodotti e servizi IoT. Desideri saperne di più?

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# IDENTITÀ STUDENTE — Per personalizzare il notebook
# ═══════════════════════════════════════════════════════════════════

# Scrivi il tuo nome qui (tra le virgolette)
NOME_STUDENTE = ""  # ← SCRIVI IL TUO NOME

# Verifica se il nome è stato inserito
if NOME_STUDENTE:
    # Se il nome non è vuoto, stampa un messaggio di benvenuto
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    # Se il nome è vuoto, avvisa lo studente
    print("⚠️ Scrivi il tuo nome!")


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ESERCIZIO 1 — Indicizza un documento tuo ★☆☆
# ═══════════════════════════════════════════════════════════════════
# Consegna: 
#   Crea un documento di testo su un argomento a tua scelta 
#   (può essere anche una dispensa del corso, una ricetta, un regolamento).
#   Indicizzalo in ChromaDB e fai 3 domande. 
#   I chunk recuperati sono rilevanti?
#
# Valutazione: 2 punti se il codice è completo e i risultati sono sensati

# PASSO 1: Crea il tuo documento
# Scrivi il testo del tuo documento tra i tripli apici (""")
# Almeno 300 caratteri per avere abbastanza informazione da indicizzare
mio_documento = """
# Scrivi qui il tuo documento (almeno 300 caratteri)
"""

# PASSO 2: Crea una nuova collection per il tuo documento
# Usiamo un nome diverso per non sovrascrivere la collection WiData
# Decommentare la linea sottostante quando sei pronto
# mia_collection = chroma_client.get_or_create_collection(name="mio_doc")

# PASSO 3: Chunka e indicizza il tuo documento
# Applica la funzione chunka_testo() al tuo documento
# poi aggiungi i chunk alla collection con .add()
# chunks = chunka_testo(mio_documento)
# mia_collection.add(
#     documents=chunks,
#     ids=[f"chunk_{i}" for i in range(len(chunks))]
# )

# PASSO 4: Fai 3 domande al tuo documento
# Usa una funzione simile a cerca() per recuperare chunk dalla tua collection
# e stampa i risultati
# domanda1 = "Scrivi una domanda rilevante per il tuo documento"
# risultati1 = mia_collection.query(query_texts=[domanda1], n_results=2)
# etc.

print("⏳ Completa l'esercizio seguendo i 4 passi sopra")


### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ESERCIZIO 2 — Sperimenta con il chunking ★★☆
# ═══════════════════════════════════════════════════════════════════
# Consegna:
#   Prova a indicizzare lo stesso documento con:
#   - chunk_size=200 (piccoli)
#   - chunk_size=400 (medi)
#   - chunk_size=800 (grandi)
#
#   Per la stessa domanda, i chunk recuperati sono diversi? 
#   Quale dimensione dà risultati migliori?
#
# Valutazione: 3 punti se testiamo almeno 3 dimensioni con la stessa domanda
#             e commentiamo quale funziona meglio

# Scegli una domanda di test dal tuo documento dell'Esercizio 1
domanda_test = "Inserisci qui una domanda sul tuo documento"  # ← modifica

# Ciclo: per ogni dimensione di chunk, ricrea l'indice e fai la ricerca
for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)
    
    # TODO: Implementa i seguenti step:
    # 1. Crea una nuova collection (con nome diverso per ogni chunk_size)
    #    col_name = f"doc_chunk_{chunk_size}"
    #    collection_test = chroma_client.get_or_create_collection(name=col_name)
    
    # 2. Chunka il documento con la dimensione corrente
    #    chunks = chunka_testo(mio_documento, chunk_size=chunk_size)
    
    # 3. Indicizza i chunk
    #    collection_test.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
    
    # 4. Fai la ricerca
    #    risultati = collection_test.query(query_texts=[domanda_test], n_results=2)
    
    # 5. Stampa i risultati
    #    for chunk in risultati["documents"][0]:
    #        print(f"  {chunk[:150]}...\n")
    
    print("  [Implementa il codice qui]")

# Alla fine, scrivi un commento: quale chunk_size ha dato i risultati migliori?
print("\n" + "="*50)
print("ANALISI:")
print("="*50)
# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: ...


### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ESERCIZIO 3 — RAG + Conversazione multi-turno ★★☆
# ═══════════════════════════════════════════════════════════════════
# Consegna:
#   Integra RAG nella funzione chat() della Lezione 3 (quella con la history).
#   Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.
#
#   La funzione deve:
#   1. Recuperare chunk rilevanti con cerca()
#   2. Includere il contesto RAG nel messaggio
#   3. Mantenere la history della conversazione
#   4. Usare la history per capire il contesto multi-turno
#
# Valutazione: 4 punti se il chatbot mantiene sia RAG che history

# Inizializza la history come lista vuota (salva la conversazione)
history = []

def chat_rag_con_storia(domanda):
    """
    Chatbot con RAG + conversazione multi-turno.
    
    Questo è come la funzione chat() della Lezione 3, 
    ma arricchita con il contesto RAG dai documenti.
    """
    # TODO: Implementa i seguenti step:
    
    # STEP 1: Recupera chunk rilevanti
    #   chunks_rilevanti = cerca(domanda, n_risultati=3)
    #   contesto = "\n\n---\n\n".join(chunks_rilevanti)
    
    # STEP 2: Aggiungi il messaggio dell'utente alla history
    #   history.append({"role": "user", "content": domanda})
    
    # STEP 3: Costruisci il sistema prompt che include RAG
    #   system = f"Sei un assistente utile. Basati su questo contesto:\n{contesto}"
    
    # STEP 4: Chiama Claude con la history e il system prompt
    #   response = client.messages.create(
    #       model="claude-haiku-4-5-20251001",
    #       max_tokens=800,
    #       system=system,
    #       messages=history
    #   )
    
    # STEP 5: Estrai la risposta
    #   risposta = response.content[0].text
    
    # STEP 6: Aggiungi la risposta alla history
    #   history.append({"role": "assistant", "content": risposta})
    
    # STEP 7: Gestisci la sliding window (tieni ultimi MAX_MESSAGGI)
    #   MAX_MESSAGGI = 10
    #   if len(history) > MAX_MESSAGGI:
    #       history = history[-MAX_MESSAGGI:]
    
    # STEP 8: Stampa la risposta
    #   print(f"🤖 {risposta}\n")
    
    pass  # Placeholder finché non completi


# Test: domande collegate che richiedono sia RAG che memoria
# Decommentare quando il codice è completato:

# print("🤖 Chatbot RAG + Storia — Test\n")
# chat_rag_con_storia("Parlami del sensore XS200")
# chat_rag_con_storia("Qual è la sua autonomia?")
# chat_rag_con_storia("Costa molto?")
# 
# La terza domanda richiede sia:
# - RAG: per trovare il prezzo nel documento
# - MEMORIA: per capire che "quello" si riferisce al sensore XS200 della domanda 1

print("⏳ Completa l'esercizio seguendo i 8 step")


### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ESERCIZIO 4 — CHATBOT RAG COMPLETO ★★★ (DELIVERABLE!)
# ═══════════════════════════════════════════════════════════════════
# Consegna: Costruisci il chatbot completo con:
#
# ✅ RAG sul documento WiData (recupera contesto)
# ✅ Conversation history con sliding window (ricorda il contesto)
# ✅ Streaming delle risposte (mostra la risposta man mano)
# ✅ System prompt WiData con istruzione anti-hallucination
# ✅ Loop interattivo con input() (comunica con l'utente)
# ✅ Debug: stampa i chunk usati per ogni risposta
#
# Valutazione: 5 punti (CONSEGNA FINALE)
# 
# Abilità richieste:
# - RAG (retrieval + augmentation)
# - Multi-turn conversation management
# - Streaming API
# - System prompts
# - Gestione input utente


# ═══════════════════════════════════════════════════════════════════
# IMPORT E SETUP
# ═══════════════════════════════════════════════════════════════════

import chromadb  # Database vettoriale per RAG
import json      # Per manipolare dati strutturati
import os        # Per gestire variabili ambiente
from google.colab import userdata  # Per ottenere API key da Colab
import anthropic  # SDK Claude


# Recupera la chiave API
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()


# ═══════════════════════════════════════════════════════════════════
# CONFIGURAZIONE
# ═══════════════════════════════════════════════════════════════════

# System prompt WiData con istruzioni anti-hallucination
SYSTEM = """
Sei l'assistente IA di WiData Srl, azienda IoT specializzata in smart cities.

REGOLE IMPORTANTI:
1. Rispondi SOLO basandoti sui documenti forniti nel contesto
2. Se la risposta non è nel contesto, dì: "Non ho questa informazione nei miei documenti"
3. Non inventare mai specifiche tecniche, prezzi, o informazioni non nel contesto
4. Sii conciso, amichevole e professionale
5. Se l'utente chiede qualcosa di fuori dalla tua competenza, ridirigi alla tua knowledge base

TONO: Amichevole ma professionale, come un supporto tecnico specializzato.

# ← Completa il system prompt con ulteriori istruzioni se necessario
"""

# Numero massimo di messaggi da mantenere in history
# (sliding window per risparmiare token e memoria)
MAX_MESSAGGI = 10


# ═══════════════════════════════════════════════════════════════════
# FUNZIONI HELPER
# ═══════════════════════════════════════════════════════════════════

def setup_rag(testo):
    """
    Prepara il database RAG per un documento.
    
    Passi:
    1. Crea un client ChromaDB
    2. Crea una collection per i documenti
    3. Chunka il testo
    4. Indicizza i chunk
    
    Restituisce: la collection pronta per le ricerche
    """
    # TODO: Implementa
    # chroma_client = chromadb.Client()
    # collection = chroma_client.get_or_create_collection(name="widata_rag")
    # chunks = chunka_testo(testo, chunk_size=400, overlap=50)
    # collection.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
    # return collection
    pass


def cerca(domanda, collection, n=3):
    """
    Ricerca semantica nella collection.
    
    Parametri:
    - domanda: la query di ricerca
    - collection: la ChromaDB collection
    - n: numero di chunk da ritornare
    
    Restituisce: lista di chunk più rilevanti
    """
    # TODO: Implementa
    # risultati = collection.query(query_texts=[domanda], n_results=n)
    # return risultati["documents"][0]
    pass


def chat_completo(domanda, history, collection):
    """
    Chatbot RAG completo con streaming e debug.
    
    Pipeline:
    1. Recupera i chunk rilevanti (RAG retrieval)
    2. Aggiunge la domanda alla history
    3. Costruisce il sistema prompt con contesto RAG
    4. Chiama Claude con streaming
    5. Stampa la risposta man mano che arriva (streaming)
    6. Salva nella history
    7. Applica sliding window sulla history
    8. Stampa i chunk usati (debug)
    """
    # TODO: Implementa
    
    # STEP 1: Retrieval — Recupera chunk rilevanti
    # chunks_rilevanti = cerca(domanda, collection, n=3)
    # contesto = "\n\n---\n\n".join(chunks_rilevanti)
    
    # STEP 2: Augmentation — Arricchisci il sistema prompt
    # system_aumentato = f"{SYSTEM}\n\nDOCUMENTI DI RIFERIMENTO:\n{contesto}"
    
    # STEP 3: Aggiungi alla history
    # history.append({"role": "user", "content": domanda})
    
    # STEP 4: Generation — Chiama Claude con streaming
    # risposta_completa = ""
    # with client.messages.stream(
    #     model="claude-haiku-4-5-20251001",
    #     max_tokens=800,
    #     system=system_aumentato,
    #     messages=history
    # ) as stream:
    #     for text in stream.text_stream:
    #         print(text, end="", flush=True)  # Stampa in tempo reale
    #         risposta_completa += text
    
    # STEP 5: Aggiungi la risposta alla history
    # history.append({"role": "assistant", "content": risposta_completa})
    
    # STEP 6: Sliding window sulla history
    # if len(history) > MAX_MESSAGGI:
    #     history = history[-MAX_MESSAGGI:]
    
    # STEP 7: Debug — Stampa i chunk usati
    # print(f"\n\n📄 [DEBUG] Basato su {len(chunks_rilevanti)} chunk")
    
    pass


def main():
    """
    Loop principale del chatbot.
    
    Flusso:
    1. Setup RAG
    2. Loop di interazione con l'utente
    3. Leggi domanda da input()
    4. Genera risposta con chat_completo()
    5. Esci se l'utente digita 'esci'
    """
    # TODO: Implementa
    
    # collection = setup_rag(documento_widata)
    # history = []
    # print("🤖 Chatbot WiData RAG avviato!")
    # print("Domande consigliate:")
    # print("  - 'Chi è WiData?'")
    # print("  - 'Quali sensori offrite?'")
    # print("  - 'Come funziona il gateway GW500?'")
    # print("  - 'Quanto costa?'")
    # print("Digita 'esci' per terminare.\n")
    
    # while True:
    #     utente = input("Tu: ").strip()
    #     if utente.lower() == "esci":
    #         print("👋 Arrivederci! Grazie per aver usato WiData Chatbot.")
    #         break
    #     if not utente:
    #         continue
    #     
    #     print("🤖 ", end="")
    #     chat_completo(utente, history, collection)
    #     print("\n")
    
    pass


# ═══════════════════════════════════════════════════════════════════
# ESECUZIONE
# ═══════════════════════════════════════════════════════════════════

# Decommentare per eseguire il chatbot:
# main()

print("""
⏳ Esercizio 4 — Chatbot RAG Completo

Questa è la sfida finale! Completa le 3 funzioni helper:

1️⃣  setup_rag() — Inizializza ChromaDB e indicizza il documento
2️⃣  cerca() — Implementa la ricerca semantica
3️⃣  chat_completo() — Implementa la pipeline RAG completa con streaming
4️⃣  main() — Implementa il loop di interazione con l'utente

Quando tutto è implementato, decommentare main() all'ultimo per testare.

Buona fortuna! 🚀
""")


---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*